# 06 — RecBole: SASRec (sequential neural baseline)

Trains **SASRec** (transformer, sequential) and scores it through our shared
`evaluate_predictions` — the SAME harness/metrics as the classical, hybrid and LLM
methods, so the number drops straight into the UC1 comparison table.

**Requires a CUDA GPU** (Colab T4 or the Windows box — not Apple MPS). Install:
```
pip install recbole    # may pin numpy/pandas; use a fresh env if it clashes
```

**Split:** leave-one-out by time, matching `leave_last_n_out(sample, n=1)` — each
user's last interaction is the test target, the second-to-last is validation.

**Scale caveat:** ~12.6M interactions / 50k users. Fine on a T4; the 4 GB Quadro may
OOM → subsample users if needed. RecBole is **GPU (PyTorch)**, not TPU.

In [ ]:
!pip install -q recbole

In [ ]:
import os, pandas as pd
from book_recsys.data.splits import leave_last_n_out
from book_recsys.data.schema import USER, BOOK
from book_recsys.eval.harness import build_relevance, evaluate_predictions
from book_recsys.recbole_adapter.atomic import write_inter_file
from book_recsys.recbole_adapter.export import recbole_predictions

In [ ]:
sample = pd.read_parquet("artifacts/sample.parquet")
# Our shared holdout = each user's last interaction by time. Classical/LLM methods
# are scored against this same `relevance`, so the SASRec number is comparable.
train, holdout = leave_last_n_out(sample, n=1)
relevance = build_relevance(holdout)

DATASET = "goodreads"
ds_dir = f"recbole_data/{DATASET}"
os.makedirs(ds_dir, exist_ok=True)
# Write the FULL data; RecBole reproduces leave-one-out itself (LS split in the next
# cell), so its internal test == our `holdout`. Do NOT pre-split here.
write_inter_file(sample, f"{ds_dir}/{DATASET}.inter")

In [ ]:
from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.utils import get_model, get_trainer
from recbole.utils.case_study import full_sort_topk

def run_model(model_name, extra=None):
    cfg = {
        "data_path": "recbole_data", "dataset": DATASET,
        "USER_ID_FIELD": "user_id", "ITEM_ID_FIELD": "item_id",
        "RATING_FIELD": "rating", "TIME_FIELD": "timestamp",
        "load_col": {"inter": ["user_id", "item_id", "rating", "timestamp"]},
        # Leave-one-out by time == our leave_last_n_out(n=1): last item -> test,
        # second-to-last -> validation. `mode: full` ranks against the WHOLE catalog
        # (matches our harness — NOT sampled negatives, which would inflate metrics).
        "eval_args": {"split": {"LS": "valid_and_test"}, "order": "TO",
                      "group_by": "user", "mode": "full"},
        "MAX_ITEM_LIST_LENGTH": 50,   # sequence cap for SASRec
        "epochs": 50, "topk": 10,
        # Speed knobs for the Kaggle/Colab GPU run at this scale (12.6M interactions):
        "train_batch_size": 4096,     # bigger batches = faster GPU throughput
        "eval_step": 5,               # validate every 5 epochs, not every one — full-sort
                                      # over 468k items is the wall-clock wildcard
        "stopping_step": 3,           # early-stop patience, in eval steps
        "metrics": ["NDCG", "Recall", "MRR"], "valid_metric": "NDCG@10",
    }
    cfg.update(extra or {})
    config = Config(model=model_name, dataset=DATASET, config_dict=cfg)
    dataset = create_dataset(config)
    train_data, valid_data, test_data = data_preparation(config, dataset)
    model = get_model(config["model"])(config, train_data.dataset).to(config["device"])
    trainer = get_trainer(config["MODEL_TYPE"], config["model"])(config, model)
    trainer.fit(train_data, valid_data)

    # Export top-K per test user, map RecBole's internal ids back to our ids, then
    # score through OUR harness against `relevance` (the shared leave-one-out holdout).
    internal_users = list(range(1, dataset.user_num))  # skip [PAD]=0
    _, topk_iid = full_sort_topk(internal_users, model, test_data, k=10,
                                 device=config["device"])
    uid2token = {i: dataset.id2token(dataset.uid_field, i) for i in internal_users}
    iid2token = {int(i): dataset.id2token(dataset.iid_field, int(i))
                 for i in topk_iid.reshape(-1).unique().tolist()}
    preds = recbole_predictions(internal_users, topk_iid.tolist(), uid2token, iid2token)
    return evaluate_predictions(preds, relevance, k=10)

results = {}
results["SASRec"] = run_model("SASRec")
# Optional second neural paradigm, same harness/config — uncomment if GPU time allows:
# results["LightGCN"] = run_model("LightGCN")   # graph CF (non-sequential)
pd.DataFrame(results).T

## Results

The DataFrame above (recall@10 / ndcg@10 / mrr) feeds `reports/model_report.md`,
in the same UC1 table as svd / hybrid / content / llm.

**Now aligned:** RecBole's internal leave-one-out test == our `holdout`, and
`mode: "full"` ranks against the whole catalog — so SASRec is measured on the exact
ruler as every other method. **Sanity-check after the first run:** pick one user and
confirm `uid2token` / `iid2token` map RecBole's internal ids back to user/book ids you
recognize (this is the riskiest seam; the adapter unit tests cover the mapping logic).